# Module 6: Generative AI - Retrieval-Augmented Generation (RAG)

Large Language Models are powerful, but they have limitations:
1. **Knowledge Cutoff**: They cannot answer questions about events after their training data was collected.
2. **Private Data**: They do not know about your private company files or personal notes.
3. **Hallucination**: They sometimes make up convincing facts.

**Retrieval-Augmented Generation (RAG)** solves this by retrieving relevant text snippets from a document library (vector database) and inserting them into the LLM prompt as context.

### RAG Workflow:
1. **Chunk**: Break documents into small paragraphs.
2. **Embed**: Convert chunks into numeric vectors representing semantic meaning.
3. **Store**: Save embeddings in a **Vector Database** (e.g. ChromaDB).
4. **Query**: When a user asks a question, embed their query and find the closest matching chunks.
5. **Generate**: Feed retrieved chunks + user query into the LLM prompt to get a fact-grounded answer.

> [!NOTE]
> **No API Key Required**: To make this notebook runnable out-of-the-box for beginners, we implement a **Simulated Embedding and LLM System**. You can easily swap in real OpenAI or Gemini API calls as detailed inside.

In [ ]:
import numpy as np
import chromadb
from chromadb.utils import embedding_functions

---

## 1. Setup Sample Knowledge Base

Let's define a mock database containing private information about a fictional company product: **"AeroFly Drone X1"**.

In [ ]:
documents = [
    "The AeroFly Drone X1 has a maximum battery life of 42 minutes under optimal wind conditions.",
    "AeroFly Drone X1 features a 4K camera with 3-axis mechanical stabilization and active tracking.",
    "To calibrate the compass on AeroFly Drone X1, rotate the drone 360 degrees horizontally and then 360 degrees vertically.",
    "The maximum control range for the AeroFly Drone X1 remote controller is 10 kilometers (6.2 miles).",
    "AeroFly Drone X1 is equipped with obstacle avoidance sensors on the front, back, and bottom of the chassis."
]

doc_ids = [f"doc_{i}" for i in range(len(documents))]

---

## 2. Setting Up ChromaDB Vector Database

We will initialize a local in-memory ChromaDB client. For the embedding function, we will use a lightweight, local helper to generate vector representations of our text.

In [ ]:
# Initialize Chroma DB client in memory
chroma_client = chromadb.Client()

# Create a collection to store embeddings (uses default SentenceTransformer model locally)
# Note: On first run, it might download a small open-source embedding model
collection = chroma_client.create_collection(name="aerofly_manual")

# Add documents to our vector database
collection.add(
    documents=documents,
    ids=doc_ids
)

print("Collection created and documents added to vector database!")

---

## 3. Retrieval: Querying the Database

When a user asks: **"How do I fix the compass?"**, the database calculates the cosine similarity between the query embedding and the document embeddings to retrieve the most relevant text chunks.

In [ ]:
user_query = "How long can the drone fly and how do I calibrate the compass?"

# Retrieve top 2 most semantically similar documents
results = collection.query(
    query_texts=[user_query],
    n_results=2
)

retrieved_docs = results['documents'][0]
print(f"User Query: '{user_query}'\n")
print("Retrieved Documents:")
for i, doc in enumerate(retrieved_docs):
    print(f"{i+1}. {doc}")

---

## 4. Generation: Building the RAG Prompt

We assemble the retrieved text as context, define the instructions, and prompt the LLM. 

Here we show a simulated LLM generation logic. If you have an OpenAI or Gemini API key, you can swap it in below.

In [ ]:
def construct_prompt(query, context_list):
    context = "\n".join([f"- {c}" for c in context_list])
    
    prompt = f"""You are an assistant for AeroFly customer support.
Use the following retrieved context to answer the customer's question.
If you don't know the answer or if it's not in the context, say that you don't know.

Context:
{context}

Question: {query}
Answer:
"""
    return prompt

rag_prompt = construct_prompt(user_query, retrieved_docs)
print("=== Constructed RAG Prompt ===")
print(rag_prompt)

### Calling the LLM (Simulated vs. Real API)

In [ ]:
# Simulated LLM Responder (for offline usage)
def call_simulated_llm(prompt):
    # A simple parser simulating factual generation based on context
    response = "According to the context: \n"
    if "battery life" in prompt or "long can" in prompt:
        response += "- The AeroFly Drone X1 has a maximum battery life of 42 minutes under optimal wind conditions.\n"
    if "compass" in prompt or "calibrate" in prompt:
        response += "- To calibrate the compass, you should rotate the drone 360 degrees horizontally and then 360 degrees vertically."
    return response

print("=== LLM Output ===")
print(call_simulated_llm(rag_prompt))

# =========================================================================
# Real OpenAI Example (Uncomment and run if you have an API key)
# =========================================================================
# import openai
# openai.api_key = "your-api-key-here"
# response = openai.chat.completions.create(
#     model="gpt-3.5-turbo",
#     messages=[{"role": "user", "content": rag_prompt}]
# )
# print(response.choices[0].message.content)
# =========================================================================

---

## 🏋️ Exercise: Test Hallucination vs. RAG

If you ask an LLM about **"AeroFly Drone X1 water resistance rating"**, it might guess or hallucinate an answer (e.g. "IPX4").

1. Query the vector database for: `"Is the AeroFly Drone X1 waterproof?"`
2. Observe what context is retrieved.
3. See if your prompt guidelines correctly force the LLM to output "I don't know the answer as it is not in the context" instead of hallucinating.

In [ ]:
# Query database for water resistance
query_water = "Is the AeroFly Drone X1 waterproof?"

# YOUR CODE HERE